In [1]:
from pathlib import Path
import pandas as pd


In [2]:

carpeta = Path(".")

archivo_catalogo = carpeta / "formulario_limpio_1.xlsx"
sheet_catalogo = "Catálogo VRAI"

archivos_output = [
    f for f in carpeta.glob("output_*.xlsx")
    if f.name != archivo_catalogo.name
]

# Catálogo VRAI: mapeo por NRC
df_catalogo = pd.read_excel(archivo_catalogo, sheet_name=sheet_catalogo, dtype=str)

columnas_catalogo = [
    "NRC",
    "Sigla",
    "Horario Cátedra/Clase",
    "Horario Ayudantía",
    "Horario Laboratorio",
    "Horario Taller",
    "Horario Terreno",
]

mapa_catalogo = (
    df_catalogo[columnas_catalogo]
    .dropna(subset=["NRC"])
    .assign(NRC=lambda x: x["NRC"].astype(str).str.strip())
    .drop_duplicates(subset=["NRC"], keep="first")
)

# Archivos output: filas a recopilar
columnas_output = [
    "RUT UC",
    "Nombre",
    "Email",
    "NRC",
    "Unidad Académica",
]

dfs = []

for archivo in archivos_output:
    df = pd.read_excel(archivo, dtype=str, sheet_name="Con éxito")

    df = df[columnas_output].copy()
    df["NRC"] = df["NRC"].astype(str).str.strip()

    dfs.append(df)

df_final = pd.concat(dfs, ignore_index=True)

# Cruce por NRC
df_final = df_final.merge(mapa_catalogo, on="NRC", how="left")

# Orden final
df_final = df_final[
    [
        "RUT UC",
        "Nombre",
        "Email",
        "NRC",
        "Unidad Académica",
        "Sigla",
        "Horario Cátedra/Clase",
        "Horario Ayudantía",
        "Horario Laboratorio",
        "Horario Taller",
        "Horario Terreno",
    ]
]

In [3]:
df_final

,RUT UC,Nombre,Email,NRC,Unidad Académica,Sigla,Horario Cátedra/Clase,Horario Ayudantía,Horario Laboratorio,Horario Taller,Horario Terreno
0,118606K,Elaine Victoria Garcia,elaine.garcia@tufts.edu,10169,Agronomia,AGC220,CLAS = M:1220 a 1720;,NaN,NaN,NaN,NaN
1,1185799,Mathilde Marie Blandine Lagarde,mathilde.lagrd@gmail.com,10169,Agronomia,AGC220,CLAS = M:1220 a 1720;,NaN,NaN,NaN,NaN
2,118539K,Vadim MAURER-DECOUT,vadim.maurerdecout@gmail.com,10169,Agronomia,AGC220,CLAS = M:1220 a 1720;,NaN,NaN,NaN,NaN
3,1186299,Ava Louise Giere,a.giere@wustl.edu,10218,Letras,LET1005,CLAS = L:1100 a 1210; W:1100 a 1210;,NaN,NaN,TAL = V:0940 a 1050;,NaN
4,1184148,Clara Fornés Roig,fornesroigc@gmail.com,10261,Estetica,EST210A,CLAS = M:1450 a 1720;,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
506,1184598,Julian Staeva-Vieira,julian.staeva_vieira@tufts.edu,29824,Agronomia,AGC209,CLAS = L:1100 a 1330;,NaN,NaN,NaN,NaN
507,1186892,Joana Haristoy,joana.haristoy@opendeusto.es,30100,Escuela de Medicina,MED855,CLAS = J:1100 a 1330;,NaN,NaN,NaN,NaN
508,1184520,Leslie Santiago Andres,lc23-0707@lclark.edu,30374,Ciencia Politica,ICP0720,CLAS = L:1220 a 1330; W:1220 a 1330;,NaN,NaN,NaN,NaN
509,1185748,Ximena Herrera Monreal,A01705214@itesm.mx,33476,Derecho,DNO158,CLAS = V:0820 a 1050;,NaN,NaN,NaN,NaN


In [4]:

# Guardar resultado
df_final.to_excel("recopilado_outputs.xlsx", index=False)